# Stage 1 — NTP 사전학습 실습

**Next Token Prediction(NTP)** 으로 AutoGaze의 기본 가이즈 능력을 학습하는 과정을 단계별로 실습합니다.

## NTP가 하는 일

사람이 직접 레이블링한 GT 가이즈 시퀀스(`gazing_labels.json`)를 교사 신호로 삼아,  
AutoGaze의 AR 디코더가 **다음에 볼 패치 인덱스**를 예측하도록 학습합니다.  
수식으로는 다음 크로스 엔트로피 손실을 최소화합니다:

$$\mathcal{L}_{\text{NTP}} = -\sum_t \log p(a_t^* \mid a_{<t}, v)$$

여기서 $a_t^*$ 는 GT 가이즈 토큰, $v$ 는 비디오 입력입니다.

**다루는 내용**
1. 사전 준비 및 경로 설정
2. 학습 데이터 탐색 (gazing_labels.json 구조 분석)
3. GT 가이즈 시퀀스 시각화
4. Hydra 학습 설정 리뷰
5. 학습 실행 (단일 GPU / 소규모 테스트)
6. 체크포인트 구조 확인
7. 학습된 모델 인퍼런스
8. 모델 비교 (사전학습 전 vs NTP 후)

**사전 조건**
```bash
# 1. 가중치 다운로드
bash scripts/download_models.sh

# 2. 학습 데이터 다운로드 (전체 또는 서브셋)
bash scripts/download_data.sh InternVid   # ~130 GB, 소규모 실험용
```

---
## 0. 환경 확인 및 경로 설정

In [ ]:
import platform, sys
import matplotlib
import matplotlib.font_manager as fm
import torch

# 한글 폰트 설정
def _setup_korean_font():
    _sys = platform.system()
    if _sys == 'Darwin':
        matplotlib.rcParams['font.family'] = 'AppleGothic'
        _font = 'AppleGothic'
    elif _sys == 'Windows':
        matplotlib.rcParams['font.family'] = 'Malgun Gothic'
        _font = 'Malgun Gothic'
    else:
        # NanumGothic 우선 (sudo apt-get install fonts-nanum && fc-cache -fv)
        _available = {f.name for f in fm.fontManager.ttflist}
        _preferred = ['NanumGothic', 'NanumBarunGothic', 'NanumGothicCoding', 'NanumMyeongjo']
        _candidates = [f for f in _preferred if f in _available]
        if not _candidates:
            _candidates = [f.name for f in fm.fontManager.ttflist
                           if any(k in f.name for k in ('Nanum', 'UnDotum', 'Baekmuk', 'Gothic'))]
        if _candidates:
            matplotlib.rcParams['font.family'] = _candidates[0]
            _font = _candidates[0]
        else:
            print("⚠  한글 폰트 없음 — sudo apt-get install fonts-nanum && fc-cache -fv")
            return
    matplotlib.rcParams['axes.unicode_minus'] = False
    print(f"[폰트] {_font}")

_setup_korean_font()
print(f"Python : {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"Device : {'CUDA' if torch.cuda.is_available() else 'MPS' if torch.backends.mps.is_available() else 'CPU'}")

In [ ]:
from pathlib import Path
import json, subprocess, shlex

# ── 경로 설정 (환경에 맞게 수정) ─────────────────────────────────────
ROOT        = Path("..")                        # AutoGaze 프로젝트 루트
DATA_ROOT   = ROOT / "data/AutoGaze-Training-Data"
VIDEOMAE_PT = ROOT / "weights/VideoMAE_AutoGaze/videomae.pt"
EXP_NAME    = "ntp_tutorial"                    # 체크포인트 저장 이름
EXP_DIR     = ROOT / f"exps/{EXP_NAME}"

# 데이터셋 서브셋 — 실제 경로가 없으면 다운로드 먼저 필요
DATASET_PATHS = [
    DATA_ROOT / "InternVid_res448_250K",
    # DATA_ROOT / "100DoH_res448_250K",   # 주석 해제 시 추가
    # DATA_ROOT / "Ego4D_res448_250K",
    # DATA_ROOT / "scanning_SAM_res448_50K",
    # DATA_ROOT / "scanning_idl_res448_50K",
]
GAZING_LABELS = DATA_ROOT / "gazing_labels.json"

# 존재 여부 확인
checks = {
    "VideoMAE 가중치": VIDEOMAE_PT,
    "gazing_labels.json": GAZING_LABELS,
}
for label, path in checks.items():
    status = "✓" if path.exists() else "✗ (없음)"
    print(f"  {label:25s}: {status}  ({path})")

print()
for p in DATASET_PATHS:
    status = "✓" if p.exists() else "✗ (없음 — download_data.sh 필요)"
    print(f"  데이터셋 {p.name:30s}: {status}")

---
## 1. 학습 데이터 탐색

### 1-A. gazing_labels.json 구조 분석

`gazing_labels.json`은 **NTP 학습의 핵심 교사 신호**입니다.  
각 비디오에 대해 프레임별로 "어떤 패치를 봐야 하는가"가 사전 계산된 형태로 저장됩니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

if not GAZING_LABELS.exists():
    print("gazing_labels.json 이 없습니다. 학습 데이터를 먼저 다운로드하세요.")
    print("  bash scripts/download_data.sh InternVid")
else:
    with open(GAZING_LABELS) as f:
        labels = json.load(f)

    keys = list(labels.keys())
    print(f"전체 비디오 수  : {len(labels):,}")
    print(f"\n예시 키 (처음 5개):")
    for k in keys[:5]:
        print(f"  {k}")

    # 구조 상세
    example_key = keys[0]
    entry = labels[example_key]
    print(f"\n─── {example_key} ───")
    print(f"  프레임 수       : {len(entry['gazing_pos'])}")
    print(f"  gazing_pos[0]  : {entry['gazing_pos'][0]}  ({len(entry['gazing_pos'][0])} 패치)")
    print(f"  gazing_pos[1]  : {entry['gazing_pos'][1]}  ({len(entry['gazing_pos'][1])} 패치)")
    print(f"  task_losses[0] : {entry['task_losses'][0]}")

In [ ]:
if GAZING_LABELS.exists():
    # 전체 데이터셋 통계
    all_counts = []      # 프레임별 패치 수 (전체 비디오)
    frame_counts = []    # 비디오별 프레임 수

    for entry in labels.values():
        frame_counts.append(len(entry['gazing_pos']))
        for frame_pos in entry['gazing_pos']:
            all_counts.append(len(frame_pos))

    all_counts  = np.array(all_counts)
    frame_counts = np.array(frame_counts)

    print("=== gazing_labels.json 통계 ===")
    print(f"비디오 수        : {len(labels):,}")
    print(f"총 프레임 수     : {len(all_counts):,}")
    print(f"비디오당 프레임  : {frame_counts.mean():.1f} ± {frame_counts.std():.1f}")
    print(f"프레임당 GT 패치 : {all_counts.mean():.1f} ± {all_counts.std():.1f}")
    print(f"  최솟값: {all_counts.min()}, 최댓값: {all_counts.max()}")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist(all_counts, bins=50, color='steelblue', edgecolor='white')
    axes[0].axvline(all_counts.mean(), color='tomato', linestyle='--',
                    label=f'평균 {all_counts.mean():.1f}')
    axes[0].set_xlabel('프레임당 GT 가이즈 패치 수')
    axes[0].set_ylabel('빈도')
    axes[0].set_title('GT 가이즈 패치 수 분포')
    axes[0].legend()

    axes[1].hist(frame_counts, bins=30, color='seagreen', edgecolor='white')
    axes[1].set_xlabel('비디오당 프레임 수')
    axes[1].set_ylabel('빈도')
    axes[1].set_title('비디오 길이 분포 (프레임 수)')

    plt.tight_layout()
    plt.show()

### 1-B. 데이터셋 비디오 샘플 확인

In [ ]:
# 사용 가능한 데이터셋의 비디오 수 확인
for ds_path in DATASET_PATHS:
    if not ds_path.exists():
        print(f"  {ds_path.name}: 없음 (스킵)")
        continue

    mp4_train = list((ds_path / 'train').rglob('*.mp4')) if (ds_path / 'train').exists() else []
    mp4_val   = list((ds_path / 'val').rglob('*.mp4'))   if (ds_path / 'val').exists()   else []
    total_size_gb = sum(f.stat().st_size for f in mp4_train + mp4_val) / 1e9

    print(f"  {ds_path.name}")
    print(f"    train: {len(mp4_train):,} 비디오")
    print(f"    val  : {len(mp4_val):,} 비디오")
    print(f"    총 크기: {total_size_gb:.1f} GB")

---
## 2. GT 가이즈 시퀀스 시각화

`gazing_labels.json`에 저장된 GT 가이즈 패치를 비디오 프레임 위에 시각화합니다.

In [ ]:
import av
import sys; sys.path.insert(0, "..")
from autogaze.datasets.video_utils import read_video_pyav, process_video_frames
import matplotlib.patches as mpatches

# gazing_labels.json 에서 실제 비디오 경로 찾기
def find_video_for_key(key, dataset_paths):
    """JSON 키에서 비디오 경로 탐색 (마지막 3레벨 매칭)."""
    for ds in dataset_paths:
        if not ds.exists():
            continue
        # key 형식: 'InternVid_res448_250K/train/xxx.mp4'
        parts = Path(key).parts
        candidate = ds.parent / Path(*parts) if len(parts) >= 2 else ds / key
        if candidate.exists():
            return candidate
        # ds 이름으로도 시도
        candidate2 = ds.parent / key
        if candidate2.exists():
            return candidate2
    return None

def visualize_gt_gaze(video_path, gt_frames_patches, num_frames=8,
                      patch_grid=14, frame_size=224):
    """
    GT 가이즈 패치를 비디오 프레임에 시각화.
    gt_frames_patches: list of list of int (프레임별 패치 인덱스)
    """
    container = av.open(str(video_path))
    total = container.streams.video[0].frames
    indices = list(range(min(num_frames, total)))
    raw = read_video_pyav(container, indices)
    container.close()
    raw = process_video_frames(raw, len(indices))

    T = len(indices)
    fig, axes = plt.subplots(2, T, figsize=(T * 2.5, 5))

    for t in range(T):
        # 상단: 원본
        axes[0, t].imshow(raw[t])
        axes[0, t].set_title(f'F{t+1} 원본', fontsize=8)
        axes[0, t].axis('off')

        # 하단: GT 가이즈 마스크
        mask = np.zeros((patch_grid, patch_grid), dtype=np.float32)
        if t < len(gt_frames_patches):
            for idx in gt_frames_patches[t]:
                if 0 <= idx < patch_grid * patch_grid:
                    r, c = divmod(idx, patch_grid)
                    mask[r, c] = 1.0

        import torch
        import torch.nn.functional as F
        mask_up = F.interpolate(
            torch.from_numpy(mask).unsqueeze(0).unsqueeze(0),
            size=(frame_size, frame_size), mode='nearest'
        ).squeeze().numpy()

        # 원본 리사이즈 + 마스크 오버레이
        from PIL import Image
        orig_resized = np.array(Image.fromarray(raw[t]).resize((frame_size, frame_size)))
        dim = 0.25
        overlay = (orig_resized / 255.0) * (dim + (1 - dim) * mask_up[:, :, None])
        axes[1, t].imshow(np.clip(overlay, 0, 1))

        # 패치 경계
        px = frame_size // patch_grid
        for pi in range(patch_grid):
            for pj in range(patch_grid):
                if mask[pi, pj] > 0.5:
                    axes[1, t].add_patch(mpatches.Rectangle(
                        (pj * px - 0.5, pi * px - 0.5), px, px,
                        lw=0.8, edgecolor='lime', facecolor='none'
                    ))
        axes[1, t].set_title(f'GT 가이즈 ({len(gt_frames_patches[t]) if t < len(gt_frames_patches) else 0}패치)', fontsize=8)
        axes[1, t].axis('off')

    plt.suptitle(f'GT 가이즈 시퀀스 — {Path(video_path).name}', fontsize=11)
    plt.tight_layout()
    plt.show()

print("시각화 함수 준비 완료 ✓")

In [ ]:
if GAZING_LABELS.exists() and any(p.exists() for p in DATASET_PATHS):
    # gazing_labels.json 키 중 실제 파일이 있는 것 탐색
    sample_key = None
    sample_video = None
    for k in list(labels.keys())[:50]:
        v = find_video_for_key(k, DATASET_PATHS)
        if v:
            sample_key = k
            sample_video = v
            break

    if sample_video:
        print(f"샘플 비디오: {sample_video}")
        gt_patches = labels[sample_key]['gazing_pos']
        visualize_gt_gaze(sample_video, gt_patches, num_frames=8, patch_grid=14)
    else:
        print("gazing_labels.json 키에 매칭되는 비디오 파일을 찾지 못했습니다.")
        print("데이터 경로를 확인하거나, 아래에서 직접 비디오 경로를 지정하세요.")
else:
    # 대신 예제 비디오로 AutoGaze 출력을 'GT'처럼 시각화
    print("데이터가 없으므로 assets/example_input.mp4 + 임의 패치로 시각화합니다.")
    example_video = ROOT / 'assets/example_input.mp4'
    if example_video.exists():
        dummy_gt = [[np.random.randint(0, 196) for _ in range(20)] for _ in range(8)]
        visualize_gt_gaze(example_video, dummy_gt, num_frames=8, patch_grid=14)

---
## 3. Hydra 학습 설정 리뷰

AutoGaze는 [Hydra](https://hydra.cc/)를 통해 설정을 관리합니다.  
`autogaze/configs/` 아래 YAML 파일들이 학습 설정을 정의합니다.

In [ ]:
import yaml

# NTP 설정 파일 확인
configs_dir = ROOT / 'autogaze/configs'

def list_configs():
    print("autogaze/configs/ 구조:")
    for p in sorted(configs_dir.rglob('*.yaml')):
        print(f"  {p.relative_to(configs_dir)}")

list_configs()

In [ ]:
# NTP 학습에 사용되는 핵심 파라미터 설명
ntp_params = {
    "dataset": {
        "clip_len": (16, "프레임 수"),
        "gt_gazing_pos_paths.train": ("gazing_labels.json 경로", "NTP 전용"),
    },
    "model": {
        "gazing_ratio_config.fixed.gazing_ratio": (0.1, "NTP는 낮은 비율로 학습"),
        "scales": ("32+64+112+224", "멀티스케일 패치"),
        "num_vision_tokens_each_frame": (265, "프레임당 토큰 수"),
        "gaze_decoder_config.num_multi_token_pred": (10, "병렬 예측 토큰 수"),
    },
    "task": {
        "recon_model": ("facebook/vit-mae-large", "재건 모델"),
        "recon_model_config.loss_type": ("l1+dinov2_reg+siglip2", "복합 재건 손실"),
        "recon_model_config.loss_weights": ("1+0.3+0.3", "손실 가중치"),
    },
    "trainer": {
        "lr": (5e-4, "학습률"),
        "n_epochs": (150, "논문 설정 / 테스트는 1"),
        "batch_size": (1024, "8 GPU / 단일 GPU는 32"),
        "train_gaze": (True, "가이즈 모델 학습"),
        "train_task": (False, "VideoMAE 동결"),
        "detach_task": (True, "VideoMAE no_grad"),
    },
}

for group, params in ntp_params.items():
    print(f"\n[{group}]")
    for key, (val, desc) in params.items():
        print(f"  {key:<50s} = {str(val):<20s} # {desc}")

---
## 4. 학습 실행

### 4-A. 소규모 테스트 (빠른 동작 확인)

전체 데이터와 150 에폭 학습은 매우 오래 걸립니다.  
여기서는 **10 스텝**만 실행해 파이프라인이 정상 동작하는지 확인합니다.

In [ ]:
import subprocess, shlex, time

def run_training(cmd, cwd=None, timeout=600):
    """학습 명령 실행 + 실시간 출력 표시."""
    cwd = cwd or str(ROOT)
    print(f"$ {cmd}\n")
    print("─" * 60)
    try:
        proc = subprocess.Popen(
            shlex.split(cmd), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, cwd=cwd
        )
        for line in proc.stdout:
            print(line, end="")
        proc.wait(timeout=timeout)
        print("─" * 60)
        print(f"종료 코드: {proc.returncode}")
        return proc.returncode
    except subprocess.TimeoutExpired:
        proc.kill()
        print(f"\n[타임아웃 {timeout}s]")
        return -1

# 데이터 경로 조합 (존재하는 것만)
valid_paths = [str(p) for p in DATASET_PATHS if p.exists()]
if not valid_paths:
    print("⚠  학습 데이터 없음 — 아래 명령으로 다운로드하세요:")
    print("   bash scripts/download_data.sh InternVid")
else:
    DATA_ROOTS_STR = ",".join(valid_paths)
    print(f"학습에 사용할 데이터셋: {DATA_ROOTS_STR}")

In [ ]:
# ── 소규모 NTP 테스트 실행 ──────────────────────────────────────────
# val_nsteps=10, save_nsteps=10 으로 설정해 빠르게 동작 확인

if valid_paths and VIDEOMAE_PT.exists() and GAZING_LABELS.exists():
    cmd = f"""
python -m autogaze.train
  --config-name video_folder_video_mae_reconstruction_ar_gaze_ntp
  dataset.root='{DATA_ROOTS_STR}'
  dataset.gt_gazing_pos_paths.train='{GAZING_LABELS}'
  dataset.clip_len=16
  model.gazing_ratio_config.sample_strategy_during_training=fixed
  model.gazing_ratio_config.sample_strategy_during_inference=fixed
  model.gazing_ratio_config.fixed.gazing_ratio=0.1
  model.scales=32+64+112+224
  model.num_vision_tokens_each_frame=265
  task.recon_model=facebook/vit-mae-large
  task.recon_sample_rate=0.125
  task.recon_model_config.loss_type=l1+dinov2_reg+siglip2
  task.recon_model_config.loss_weights=1+0.3+0.3
  task.scales=32+64+112+224
  algorithm.optimize_task_loss_prediction=True
  trainer.train_gaze=True
  trainer.train_task=False
  trainer.detach_task=True
  trainer.lr=5e-4
  trainer.n_epochs=1
  trainer.batch_size=4
  trainer.per_gpu_max_batch_size=2
  trainer.val_nsteps=10
  trainer.save_nsteps=10
  trainer.task_weights={VIDEOMAE_PT}
  trainer.exp_name={EXP_NAME}
""".replace('\n', ' ').replace('  ', ' ').strip()

    run_training(cmd, timeout=300)
else:
    print("필요한 파일이 없어 학습을 건너뜁니다.")
    print("체크: 데이터셋, VideoMAE 가중치, gazing_labels.json")

### 4-B. 실제 학습 (스크립트 방식)

전체 학습은 터미널에서 스크립트로 실행하는 것이 편리합니다.  
백그라운드로 실행하면 노트북을 닫아도 학습이 계속됩니다.

In [ ]:
# 전체 NTP 학습 스크립트 생성
script_content = f"""#!/usr/bin/env bash
# 자동 생성: 노트북에서 사용자 경로로 생성
set -euo pipefail

bash scripts/train_ntp_single_gpu.sh \\
    "{DATA_ROOTS_STR if valid_paths else '<데이터셋 경로>'}" \\
    "{VIDEOMAE_PT}"
"""

script_path = ROOT / 'scripts/my_train_ntp.sh'
script_path.write_text(script_content)
script_path.chmod(0o755)
print(f"학습 스크립트 생성: {script_path}")
print()
print("실행 방법 (터미널에서):")
print(f"  bash {script_path.relative_to(ROOT)}")
print()
print("백그라운드 실행:")
print(f"  nohup bash {script_path.relative_to(ROOT)} > ntp_train.log 2>&1 &")
print(f"  tail -f ntp_train.log")

---
## 5. 체크포인트 구조 확인

In [ ]:
# exps/ 아래 저장된 체크포인트 목록 확인
exps_dir = ROOT / 'exps'

if not exps_dir.exists():
    print("exps/ 디렉터리가 없습니다. 학습을 먼저 실행하세요.")
else:
    print("=== exps/ 구조 ===")
    for exp in sorted(exps_dir.iterdir()):
        if not exp.is_dir():
            continue
        ckpts = sorted([d for d in exp.iterdir() if d.is_dir()])
        print(f"\n{exp.name}/")
        for ckpt in ckpts:
            files = list(ckpt.iterdir())
            size_mb = sum(f.stat().st_size for f in files if f.is_file()) / 1e6
            print(f"  {ckpt.name}/  ({len(files)} 파일, {size_mb:.1f} MB)")
            for f in sorted(files)[:3]:
                print(f"    {f.name}")
            if len(files) > 3:
                print(f"    ... 외 {len(files)-3}개")

In [ ]:
# 체크포인트 내부 구조 확인
latest_gaze_ckpt = EXP_DIR / 'checkpoint_latest_gaze'

if latest_gaze_ckpt.exists():
    print(f"체크포인트: {latest_gaze_ckpt}")
    print()
    ckpt_files = list(latest_gaze_ckpt.iterdir())
    for f in sorted(ckpt_files):
        size_mb = f.stat().st_size / 1e6
        print(f"  {f.name:<40s} {size_mb:.2f} MB")

    # pytorch_model.bin 또는 .pt 파일 키 확인
    pt_files = [f for f in ckpt_files if f.suffix in ('.bin', '.pt', '.safetensors')]
    if pt_files:
        print(f"\n가중치 파일 키 샘플:")
        state = torch.load(pt_files[0], map_location='cpu')
        if isinstance(state, dict):
            keys = list(state.keys())[:10]
            for k in keys:
                v = state[k]
                print(f"  {k:<50s} {tuple(v.shape) if hasattr(v, 'shape') else type(v).__name__}")
            if len(state) > 10:
                print(f"  ... 외 {len(state)-10}개 키")
else:
    print(f"체크포인트 없음: {latest_gaze_ckpt}")
    print("학습을 먼저 실행하세요.")

---
## 6. 학습된 모델 인퍼런스

NTP로 학습한 체크포인트를 로드해 인퍼런스를 실행합니다.

In [ ]:
from autogaze.models.autogaze import AutoGaze, AutoGazeImageProcessor
from autogaze.utils import get_device
from autogaze.datasets.video_utils import (
    sample_frame_indices, transform_video_for_pytorch
)

device = get_device()

# NTP 학습 체크포인트 또는 HuggingFace 모델 로드
if latest_gaze_ckpt.exists():
    NTP_MODEL_PATH = str(latest_gaze_ckpt)
    print(f"NTP 체크포인트 로드: {NTP_MODEL_PATH}")
else:
    NTP_MODEL_PATH = "nvidia/AutoGaze"    # 폴백: 공개 가중치
    print(f"체크포인트 없음 → 공개 모델 사용: {NTP_MODEL_PATH}")

ntp_transform = AutoGazeImageProcessor.from_pretrained(NTP_MODEL_PATH)
ntp_model     = AutoGaze.from_pretrained(NTP_MODEL_PATH).to(device)
ntp_model.eval()
print("모델 로드 완료 ✓")

In [ ]:
# 예제 비디오로 인퍼런스
example_video = ROOT / 'assets/example_input.mp4'

if example_video.exists():
    import av
    container = av.open(str(example_video))
    total = container.streams.video[0].frames
    indices = sample_frame_indices(16, 1, total, False)
    raw = read_video_pyav(container, indices)
    container.close()
    raw = process_video_frames(raw, 16)

    video_t = transform_video_for_pytorch(raw, ntp_transform)[None].to(device)

    with torch.inference_mode():
        out_ntp = ntp_model(
            {"video": video_t},
            gazing_ratio=0.75,
            task_loss_requirement=0.7,
        )

    n_real = int((~out_ntp['if_padded_gazing']).sum())
    n_total = ntp_model.config.num_vision_tokens_each_frame * 16
    print(f"NTP 모델 인퍼런스 완료")
    print(f"  선택 패치: {n_real} / {n_total} ({100*n_real/n_total:.1f}%)")
    print(f"  프레임별: {out_ntp['num_gazing_each_frame'].tolist()}")
else:
    print(f"예제 비디오 없음: {example_video}")

---
## 7. 학습 로그 분석

학습 중 생성되는 TensorBoard 로그 또는 텍스트 로그를 분석합니다.

In [ ]:
# 로그 파일 탐색
log_files = list(EXP_DIR.rglob('*.log')) + list(EXP_DIR.rglob('events.out.*'))

if not log_files:
    print(f"로그 파일 없음: {EXP_DIR}")
    print("학습을 먼저 실행하세요.")
else:
    print(f"발견된 로그 파일:")
    for f in log_files:
        print(f"  {f}")

    # 텍스트 로그 파싱 (있으면)
    txt_logs = [f for f in log_files if f.suffix == '.log']
    if txt_logs:
        print(f"\n=== {txt_logs[0].name} (마지막 20줄) ===")
        lines = txt_logs[0].read_text().splitlines()
        for line in lines[-20:]:
            print(f"  {line}")

In [ ]:
# TensorBoard 실행 (있을 경우)
tb_events = list(EXP_DIR.rglob('events.out.*'))
if tb_events:
    print("TensorBoard 실행 명령:")
    print(f"  tensorboard --logdir {EXP_DIR}")
    print("  → 브라우저에서 http://localhost:6006 접속")
else:
    print("TensorBoard 이벤트 파일 없음 (학습 후 생성됩니다)")

---
## 정리

| 단계 | 핵심 내용 |
| --- | --- |
| 데이터 | `gazing_labels.json`: 비디오당 프레임별 GT 패치 인덱스 |
| 손실 | NTP 크로스 엔트로피: 다음 가이즈 토큰 예측 |
| 가이즈 비율 | NTP는 `gazing_ratio=0.1` (낮게 시작) |
| 설정 파일 | `video_folder_video_mae_reconstruction_ar_gaze_ntp` |
| 체크포인트 | `exps/<exp_name>/checkpoint_latest_gaze` → Stage 2 입력 |
| 다음 단계 | **03_train_rl_ko.ipynb** — GRPO RL 후학습으로 성능 향상 |